# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanzimul3islam/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

**Scope:** Week 1 framing, using the bundled starter CSV. No model is trained here.
Run all cells in order with Python 3 and pandas (Colab already includes pandas). Locally, run from the repo or its notebook folder. In Colab, the first cell can fetch the exact public starter CSV pinned below. Only aggregate results are displayed.

Guidance used: [lane guide](../../docs/ml-intern-dataset-and-lane-guide.md), [data dictionary](../../docs/data-dictionary.md), [framing skill](../../skills/framing-ml-problems/SKILL.md), and [FlyRank data skill](../../skills/flyrank/flyrank-data/SKILL.md).

## 1. My lane (or freestyle) and why

My provisional choice is **Lane 2: Refresh / Content Opportunity Scoring**. I want to help an editor decide which already-visible pages deserve a closer look when review time is limited. The starter data contains exposure, recent impression movement, and time since update, so I can connect a measurable pattern to a practical review decision. Section 3 finds 9,961 visible pages with a recorded downward trend, including 4,053 with no update in at least 90 days. That makes prioritization worth investigating; it does not establish that those pages need editing. I will begin with a transparent rule, check its mistakes, and only consider ML if it improves a properly validated queue. This lane can change until the end of Week 4.

In [1]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
import hashlib
import platform
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
DATA_COMMIT = "86863f1578555c069a9f4afea00fbdce6037bf75"
EXPECTED_SHA256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
local_file = next((root / DATA_PATH for root in [Path.cwd(), *Path.cwd().parents]
                   if (root / DATA_PATH).is_file()), None)
if local_file is not None:
    raw = local_file.read_bytes()
    source = "Bundled starter CSV (local checkout)"
else:
    # Colab often opens a notebook without cloning the repository.
    url = ("https://raw.githubusercontent.com/tanzimul3islam/"
           f"flyrank-ml-internship-starter/{DATA_COMMIT}/{DATA_PATH.as_posix()}")
    with urlopen(url, timeout=60) as response:
        raw = response.read()
    source = "Public starter CSV (pinned GitHub revision)"

assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256, "Dataset changed; review the framing numbers."
df = pd.read_csv(BytesIO(raw))
assert df.shape == (30000, 44)
assert df["content_id"].notna().all() and df["content_id"].is_unique
assert df["client_id"].notna().all() and df["client_id"].nunique() == 32
print(source)
print(f"Grain verified: {len(df):,} unique content items, {df.client_id.nunique()} pseudonymized clients.")
print(f"Python {platform.python_version()}; pandas {pd.__version__}")
print(f"CSV SHA-256: {EXPECTED_SHA256}")

Bundled starter CSV (local checkout)
Grain verified: 30,000 unique content items, 32 pseudonymized clients.
Python 3.12.14; pandas 2.2.3
CSV SHA-256: c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with at least 500 search impressions in the trailing 90 days, which should an editor review first for possible content refresh, considering recent visibility loss and time since update?

- **Decision and owner:** A content editor or SEO lead allocates a limited review budget. I assume a pilot capacity of 20 pages per review cycle across this sample; this is a planning assumption, not an observed team capacity.
- **Unit of analysis:** One pseudonymized content item (page) at the starter snapshot. A later warehouse version would use one client–content item at a specified decision date. This CSV has no explicit report-date column, so I cannot establish a calendar-based future test from it.
- **Output:** A ranked review queue with page/client pseudonyms, priority score, supporting measurements, reason codes, and an evidence-strength flag. The flag would indicate evidence coverage, not a probability that editing will succeed.
- **Action:** Inspect the top pages for outdated material, intent mismatch, or technical issues; also check seasonality and traffic moving to related pages. Refresh only if review supports it; otherwise monitor or investigate further. The queue will not authorize automatic rewriting, merging, or pruning.
- **Cost of a wrong call:** A false positive wastes a review slot and may lead to an unnecessary edit that harms useful content. A false negative delays attention to a page losing visibility. The data does not measure editorial hours, revenue, or treatment effects, so I will not invent a monetary return.

**Task and success measure:** Ranking / scoring. The intended success measure is **precision@20**: the share of the first 20 candidates that an independent editorial review confirms deserve action. Those judgments are not in this CSV. For an executable check today, Section 4 measures agreement with the existing `trend_direction == "down"` proxy; it is not editorial precision or a future outcome. A later prediction extension would need non-overlapping past-feature and future-outcome windows from the warehouse, plus client-aware validation. Its observed outcome could be next-window impression loss, but even that would not measure refresh benefit.

**Why data or ML helps:** There are too many pages to inspect equally. Data makes exposure, movement, and freshness comparable and gives reviewers explicit reasons. The first baseline will select visible pages and rank the longest-unupdated first, breaking ties by impressions. A simple rule may be enough. ML would earn a place only through a meaningful improvement over that rule on held-out evidence, without hiding why a page was selected. The goal is better use of review time, not simply to train a model.

In [2]:
# Provisional policy choices, declared before computing the summaries.
MIN_IMPRESSIONS = 500
STALE_DAYS = 90
REVIEW_CAPACITY = 20
needed = ["impressions_90d", "days_since_last_update", "trend_direction"]
assert df[needed].notna().all().all(), "Handle missing evidence explicitly before selecting pages."
assert df["impressions_90d"].ge(0).all()
assert df["days_since_last_update"].ge(0).all()
visible = df["impressions_90d"].ge(MIN_IMPRESSIONS)
down = df["trend_direction"].eq("down")  # Descriptive proxy only; never a model feature.
stale = df["days_since_last_update"].ge(STALE_DAYS)
print(f"Policies: exposure >= {MIN_IMPRESSIONS} impressions/90d; unupdated >= {STALE_DAYS} days; review K={REVIEW_CAPACITY}.")

Policies: exposure >= 500 impressions/90d; unupdated >= 90 days; review K=20.


## 3. Quick look at the data (2–3 real numbers)

The code below computes three nested counts directly from the starter CSV, with each denominator shown. “Visible” here is my provisional exposure floor, not a claim that a page ranks well. The dictionary defines “down” as a last-30-day versus previous-30-day impression drop greater than 20%.

**What the measurements suggest:** Of 30,000 pages, **16,726 (55.75%)** meet the exposure floor. Within that group, **9,961 (59.55%)** have the recorded downward trend; among those, **4,053 (40.69%)** have not been updated for at least 90 days. This leaves many more possible review candidates than the assumed 20-slot budget. It supports investigating a queue rather than reviewing every page equally. It does not show that age caused the decline or that any particular candidate would benefit from a refresh.

The 500-impression and 90-day cutoffs are starting policies. In the next seven weeks I will check data quality and group effects, compare alternative thresholds, inspect top and borderline cases, build a baseline, and test any more complex method on held-out evidence. I will retain the simpler method if the extra complexity does not help.

In [3]:
summary = pd.DataFrame([
    {"Measure": "Visible pages", "Pages": int(visible.sum()),
     "Denominator": len(df), "Denominator meaning": "All starter pages"},
    {"Measure": "Visible + recorded down", "Pages": int((visible & down).sum()),
     "Denominator": int(visible.sum()), "Denominator meaning": "Visible pages"},
    {"Measure": "Visible + down + unupdated >= 90 days", "Pages": int((visible & down & stale).sum()),
     "Denominator": int((visible & down).sum()), "Denominator meaning": "Visible + recorded down"},
])
assert summary["Denominator"].gt(0).all()
summary["Percent of denominator"] = (100 * summary["Pages"] / summary["Denominator"]).round(2)
print(summary.to_string(index=False))

                              Measure  Pages  Denominator     Denominator meaning  Percent of denominator
                        Visible pages  16726        30000       All starter pages                   55.75
              Visible + recorded down   9961        16726           Visible pages                   59.55
Visible + down + unupdated >= 90 days   4053         9961 Visible + recorded down                   40.69


## 4. Careful words: what I can and can't claim

I can say **“we observed a sizeable set of visible pages with a recorded impression decline; some also have older updates, making review prioritization worth testing.”** These are descriptive, directional, decision-support observations about this starter slice.

I cannot say that a refresh will recover traffic, that stale content caused the decline, that I predicted Google's algorithm, or that the findings generalize to every client. I have not evaluated a model, observed editorial judgments, or measured future performance. The starter slice is not the full warehouse.

Important limits for the next stage:

- `trend_direction` is derived from `trend_pct`, which uses the recent comparison windows. A model predicting that flag would reconstruct a proxy, not discover the correct editorial action. Neither trend field can be an input to such a model. Recent 30-day inputs and overlapping 90-day totals also require a timing audit before any forecasting claim.
- Zero average position means missing position, not rank zero. Rates are percentages (`ctr = 0.76` means 0.76%). Missing word counts or keyword fields are not zeros; their absence varies by content type.
- Client/page IDs are only for linking, grouping, and validation. Future evaluation needs client-level checks because a pooled queue may be dominated by large clients.
- The aggregate snapshot cannot separate seasonality, consolidation, short-lived noise, and persistent deterioration. A stronger analysis requires daily history and adequate tracking coverage. No raw client identities, private queries, or page URLs are exposed here.

**Metric check, not validation:** Below I run the declared freshness baseline and compute its same-snapshot agreement with the recorded downward flag. This checks that a top-K metric is computable. It is a descriptive proxy result on data already inspected, not evidence of future predictive skill, actionability, model lift, or refresh impact.

In [4]:
# Rank without using either trend field; do not print row-level data or identifiers.
eligible = df.loc[visible].sort_values(
    ["days_since_last_update", "impressions_90d"],
    ascending=[False, False], kind="stable",
)
assert len(eligible) >= REVIEW_CAPACITY
shortlist = eligible.head(REVIEW_CAPACITY)
proxy_positives = int(shortlist["trend_direction"].eq("down").sum())
proxy_precision = proxy_positives / REVIEW_CAPACITY
proxy_base_rate = eligible["trend_direction"].eq("down").mean()
print(f"Freshness baseline: {proxy_positives}/{REVIEW_CAPACITY} top pages have the recorded down flag.")
print(f"Descriptive proxy precision@{REVIEW_CAPACITY}: {proxy_precision:.2%}")
print(f"Recorded down share among all eligible pages: {proxy_base_rate:.2%}")
print("Same-snapshot proxy agreement only; independent editorial/future validation is still required.")

Freshness baseline: 16/20 top pages have the recorded down flag.
Descriptive proxy precision@20: 80.00%
Recorded down share among all eligible pages: 59.55%
Same-snapshot proxy agreement only; independent editorial/future validation is still required.


## 5. Self-check

- [x] Chose one predefined lane and explained why it is provisional.
- [x] Named the decision owner, unit of analysis, output, action, and costs of errors.
- [x] Loaded the actual starter CSV and computed three supporting counts with explicit denominators.
- [x] Explained why a plain baseline comes first and ML must earn its place.
- [x] Distinguished observed measurements and proxy agreement from actionability, forecasting, and causal claims.
- [x] Ran the notebook top to bottom with no errors; saved the executed outputs.
- [x] Displayed only aggregates; no client names, data URLs, or private queries. Repository and documentation links are public references.
- [x] Prepared the completed notebook at `work/notebooks/w01_research_question.ipynb` for the assignment commit.

**Submission:** Use the repository URL after this notebook's commit is available on GitHub. Portal submission is a separate final step.

**AI assistance:** An AI assistant drafted this framing and executed the supporting calculations using the repo's framing and data skills. The lane, review capacity, and thresholds are provisional assumptions for the intern to review and own; no stakeholder interview or manual editorial validation is claimed.